# ARTI5001 Assignment 3: Single-Speaker TTS on LJSpeech
**Group:** `<GROUP_NUMBER>`  **Date:** `<DD Month 2026>`

| Member | ID | Owns (code / report / slides) |
|---|---|---|
| `<NAME 1>` | `<ID>` | 1.1 Data & pre-processing · 1.4 attribution/env · Report 2.1 + conclusion |
| `<NAME 2>` | `<ID>` | 1.2 End-to-end pipeline · Report 2.2 + 2.5 |
| `<NAME 3>` | `<ID>` | 1.3 Experiments · 1.4 metrics · Report 2.3 + 2.4 |

### Marker quick-start
1. `pip install -r requirements.txt`
2. Set `DATA_DIR` in §0.2 (or env var `LJSPEECH_DIR`) to the folder that contains `metadata.csv` and `wavs/`.
3. **Run All.** Training cells are skipped if checkpoints already exist (`SKIP_TRAINING_IF_CKPT = True`).
4. Try your own sentence: §5 `synthesize("Your text here.", exp="expB")`, or from a terminal `python scripts/synth.py --text "..." --exp expB --out wav_examples/custom.wav`.
5. Outputs: `wav_examples/` (audio), `results/metrics.csv` (metrics), `results/figures/` (plots).

**Notebook map → rubric:** §1 → 1.1 · §2–3, §5 → 1.2 / report 2.2 · §4 → 1.3 / report 2.3 · §6 → 1.4 / report 2.4 · §7–8 → report 2.4–2.5 · §9 → 1.4 attribution

# 0. Project framing *(feeds report 2.1)*
## 0.1 Aim and scope
- **Aim:** `<one sentence: build a TTS system that ... and measure how <controlled variable> affects <quality/speed>>`
- **Scope:** single speaker (LJSpeech), `<subset size>` training utterances, `<compute>`; a full-scale model is out of scope (task sheet).
- **Success criteria:** intelligible speech (WER < `__`%), real-time capable (RTF < 1 on `<device>`), reproducible split + configs.

In [ ]:
# 0.2 Paths, seeds, global config: the only cell a marker should need to edit
import os, sys, random, json, time, hashlib, platform, csv
from pathlib import Path
import numpy as np

PROJECT_ROOT = Path(os.environ.get("A3_ROOT", Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()))
DATA_DIR     = Path(os.environ.get("LJSPEECH_DIR", PROJECT_ROOT / "data" / "LJSpeech-1.1"))
WAV_OUT_DIR  = PROJECT_ROOT / "wav_examples"
RESULTS_DIR  = PROJECT_ROOT / "results"
FIG_DIR      = RESULTS_DIR / "figures"
SPLIT_DIR    = RESULTS_DIR / "splits"
CKPT_DIR     = PROJECT_ROOT / "checkpoints"
for d in (WAV_OUT_DIR, RESULTS_DIR, FIG_DIR, SPLIT_DIR, CKPT_DIR):
    d.mkdir(parents=True, exist_ok=True)

SEED = 42
SPLIT_RATIO = (0.90, 0.05, 0.05)   # train / val / test, shared by ALL experiments
SKIP_TRAINING_IF_CKPT = True

def set_seed(seed: int = SEED):
    random.seed(seed); np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass

set_seed()
print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR    :", DATA_DIR, "| exists:", DATA_DIR.exists())

In [ ]:
# 0.3 Environment report (rubric: Python version + key packages + CPU/GPU)
import importlib
def env_report():
    info = {"python": sys.version.split()[0], "os": platform.platform(), "cpu": platform.processor() or platform.machine()}
    for pkg in ["numpy", "pandas", "soundfile", "librosa", "matplotlib", "torch", "torchaudio"]:
        try:
            info[pkg] = importlib.import_module(pkg).__version__
        except Exception:
            info[pkg] = "not installed"
    try:
        import torch
        info["cuda"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
        info["mps"] = bool(getattr(torch.backends, "mps", None) and torch.backends.mps.is_available())
    except ImportError:
        pass
    return info

ENV = env_report()
for k, v in ENV.items():
    print(f"{k:>10}: {v}")
_ = (RESULTS_DIR / "environment.json").write_text(json.dumps(ENV, indent=2))

# 1. Dataset handling & pre-processing *(Rubric 1.1, 9 marks; owner: Member 1)*
**HD checklist:** ☐ metadata + audio paths load ☐ reproducible split (ratio + seed) ☐ sample-rate policy ☐ trimming ☐ text normalisation **justified** ☐ sanity checks (stats + listen + plot)

## 1.1 Load LJSpeech
LJSpeech 1.1 (Ito & Johnson, 2017): 13,100 clips, one female speaker, ~24 h, 22,050 Hz mono 16-bit PCM.
`metadata.csv` is pipe-separated: `id | raw transcript | normalised transcript` (numbers/abbreviations expanded).

In [ ]:
import pandas as pd

def load_metadata(data_dir: Path = DATA_DIR) -> pd.DataFrame:
    # QUOTE_NONE: transcripts contain double quotes that must not be parsed as CSV quoting
    df = pd.read_csv(data_dir / "metadata.csv", sep="|", header=None, quoting=csv.QUOTE_NONE,
                     names=["id", "text_raw", "text_norm"], dtype=str, keep_default_na=False)
    df["wav_path"] = df["id"].map(lambda i: str(data_dir / "wavs" / f"{i}.wav"))
    return df

meta = load_metadata()
missing = [p for p in meta.wav_path if not Path(p).exists()]
print(f"rows: {len(meta)} | missing wavs: {len(missing)}")
meta.head()

## 1.2 Text pre-processing
**Decision:** `<e.g. use column 3 (already-expanded numbers), lowercase, normalise whitespace and quotes, keep , . ? ! for prosody>`
**Justification:** `<why: avoids training on digits the model cannot pronounce; punctuation drives pauses/intonation; ...>`
**Tokens:** `<characters / phonemes via espeak-ng (phonemizer)>`. Phonemes remove English spelling ambiguity but add a dependency.

In [ ]:
import re, unicodedata

_WS = re.compile(r"\s+")

def normalise_text(s: str) -> str:
    # Minimal cleaner in the spirit of keithito/tacotron 'english_cleaners'
    # (https://github.com/keithito/tacotron/blob/master/text/cleaners.py). TODO(M1): extend + justify
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()
    s = s.lower().replace('"', "").replace("“", "").replace("”", "")
    s = _WS.sub(" ", s).strip()
    return s

meta["text"] = meta["text_norm"].map(normalise_text)
diff = meta[meta.text_raw.str.lower() != meta.text_norm.str.lower()]
print(f"{len(diff)} utterances where raw != normalised, e.g.:")
for _, r in diff.head(3).iterrows():
    print(" RAW :", r.text_raw, "\n NORM:", r.text, "\n")
charset = sorted(set("".join(meta.text)))
print("character set:", "".join(charset))

## 1.3 Audio pre-processing policy
- **Sample rate:** `<keep 22,050 Hz: matches pretrained vocoders / resample to __ because __>`
- **Trimming:** `<librosa.effects.trim, top_db=__>`: removes leading/trailing silence that harms alignment
- **Loudness:** `<peak-normalise to -1 dBFS / none: LJSpeech is already consistent>`
- **Features (if training an acoustic model):** `<80-bin log-mel, n_fft=1024, hop=256, win=1024, fmin=0, fmax=8000>`. These must match the vocoder.

In [ ]:
import soundfile as sf, librosa

AUDIO_CFG = dict(sr=22050, top_db=40, n_fft=1024, hop_length=256, win_length=1024, n_mels=80, fmin=0, fmax=8000)

def load_audio(path, cfg=AUDIO_CFG, trim=True):
    y, sr = sf.read(path, dtype="float32")
    if sr != cfg["sr"]:
        y = librosa.resample(y, orig_sr=sr, target_sr=cfg["sr"])
    if trim:
        y, _ = librosa.effects.trim(y, top_db=cfg["top_db"])
    return y

def log_mel(y, cfg=AUDIO_CFG):
    m = librosa.feature.melspectrogram(y=y, sr=cfg["sr"], n_fft=cfg["n_fft"], hop_length=cfg["hop_length"],
                                       win_length=cfg["win_length"], n_mels=cfg["n_mels"], fmin=cfg["fmin"], fmax=cfg["fmax"])
    return np.log(np.clip(m, 1e-5, None))

y_raw, sr0 = sf.read(meta.wav_path[0], dtype="float32")
y_trim = load_audio(meta.wav_path[0])
print(f"native sr={sr0} | raw {len(y_raw)/sr0:.2f}s -> trimmed {len(y_trim)/AUDIO_CFG['sr']:.2f}s | mel shape {log_mel(y_trim).shape}")

## 1.4 Reproducible train / val / test split (shared by every experiment)

In [ ]:
def make_split(df, ratio=SPLIT_RATIO, seed=SEED):
    ids = df["id"].to_numpy().copy()
    rng = np.random.default_rng(seed)
    rng.shuffle(ids)
    n_tr, n_va = int(ratio[0] * len(ids)), int(ratio[1] * len(ids))
    return {"train": sorted(ids[:n_tr]), "val": sorted(ids[n_tr:n_tr + n_va]), "test": sorted(ids[n_tr + n_va:])}

def save_split(split, out_dir=SPLIT_DIR):
    for name, ids in split.items():
        (out_dir / f"{name}.txt").write_text("\n".join(ids) + "\n")
    digest = hashlib.md5("".join("".join(split[k]) for k in ("train", "val", "test")).encode()).hexdigest()
    (out_dir / "split_md5.txt").write_text(digest + "\n")
    return digest

SPLIT = make_split(meta)
SPLIT_MD5 = save_split(SPLIT)
assert not (set(SPLIT["train"]) & set(SPLIT["test"])) and not (set(SPLIT["train"]) & set(SPLIT["val"]))
print({k: len(v) for k, v in SPLIT.items()}, "| split md5:", SPLIT_MD5)

# Training subset (task sheet: 1-3k utterances is acceptable). Drawn ONLY from train, same seed.
def train_subset(n, seed=SEED):
    rng = np.random.default_rng(seed)
    return sorted(rng.choice(SPLIT["train"], size=n, replace=False))

## 1.5 Sanity checks: statistics, plots, listen

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Audio, display

# Durations from file headers (fast, no decoding)
meta["dur_s"] = [sf.info(p).duration for p in meta.wav_path]
meta["n_chars"] = meta.text.str.len()
meta["split"] = meta.id.map({i: k for k, ids in SPLIT.items() for i in ids})
print(meta.groupby("split")[["dur_s", "n_chars"]].describe().round(2).T)
print(f"total hours: {meta.dur_s.sum()/3600:.2f}")

fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))
ax[0].hist(meta.dur_s, bins=50); ax[0].set(title="Clip duration (s)", xlabel="s")
ax[1].hist(meta.n_chars, bins=50); ax[1].set(title="Transcript length (chars)")
ax[2].scatter(meta.n_chars, meta.dur_s, s=1, alpha=.3); ax[2].set(title="chars vs duration", xlabel="chars", ylabel="s")
plt.tight_layout(); plt.savefig(FIG_DIR / "dataset_stats.png", dpi=150); plt.show()

i = 0
print(meta.text[i]); display(Audio(load_audio(meta.wav_path[i]), rate=AUDIO_CFG["sr"]))
plt.figure(figsize=(10, 3)); plt.imshow(log_mel(load_audio(meta.wav_path[i])), origin="lower", aspect="auto")
plt.title(f"log-mel {meta.id[i]}"); plt.colorbar(); plt.savefig(FIG_DIR / "sample_mel.png", dpi=150); plt.show()

# 2. Background & method choice *(feeds report 2.2 justification; link to docs/RESEARCH.md)*
| Family | Example | Pros | Cons for our constraints |
|---|---|---|---|
| Autoregressive + vocoder | Tacotron 2 + HiFi-GAN | `<...>` | `<...>` |
| Non-autoregressive + vocoder | FastSpeech 2 + HiFi-GAN | `<...>` | `<...>` |
| End-to-end | VITS | `<...>` | `<...>` |
| Flow-matching | Matcha-TTS | `<...>` | `<...>` |

**Chosen:** `<...>` because `<constraint → decision>`.

# 3. System design *(Rubric 1.2 / report 2.2; owner: Member 2)*
## 3.1 Pipeline diagram
![pipeline](../report/figures/pipeline.png)
`text → normalise → tokenise/phonemise → acoustic model → mel [80×T] → vocoder → waveform 22.05 kHz → .wav`

## 3.3 Constraint → decision table
| Constraint | Decision | Evidence |
|---|---|---|
| Compute: `<M4 laptop / Colab T4>` | `<fine-tune pretrained / small model>` | training time in §4 |
| Time: 1 week | `<...>` | |
| Data: `<n>` utts | `<...>` | |
| Goal: intelligible + real-time | `<...>` | RTF §6.1, WER §6.2 |

In [ ]:
# 3.2 Module walk-through: print real shapes at every stage for one sentence (beyond-HD evidence)
# TODO(M2): load model, then show: text -> token ids (len) -> mel (80 x T) -> wav (samples), plus param counts per module

# 4. Controlled experiments *(Rubric 1.3, 9 marks; owner: Member 3)*
**HD checklist:** ☐ same split (md5 printed) ☐ ONE variable changed (diff printed) ☐ checkpoint selection rule ☐ validation use ☐ logging

## 4.1 Hypothesis
> Changing **`<variable>`** from `<A>` to `<B>` will `<improve/worsen>` **`<metric>`** because `<reason>`.

## 4.3 Training protocol
- Checkpoint selection: `<lowest val loss / fixed step>` · validation every `<n>` steps on `results/splits/val.txt`
- Logging: `<TensorBoard / CSV>` at `logs/<exp>/` · stopping rule: `<fixed steps = fair compute budget>`

In [ ]:
# 4.2 Experiment configs: everything not listed in the controlled variable MUST be identical
BASE_CFG = dict(model="<model>", vocoder="<vocoder>", n_train_utts=1000, steps=0, batch_size=16, lr=2e-4,
                tokens="chars", sr=AUDIO_CFG["sr"], seed=SEED, split_md5=SPLIT_MD5)
CONFIGS = {
    "expA": {**BASE_CFG},
    "expB": {**BASE_CFG, "n_train_utts": 3000},   # TODO(group): set the controlled change
}

def diff_configs(a, b):
    return {k: (a.get(k), b.get(k)) for k in sorted(set(a) | set(b)) if a.get(k) != b.get(k)}

d = diff_configs(CONFIGS["expA"], CONFIGS["expB"])
print("controlled change(s):", d)
assert len(d) == 1, "expA/expB should differ in exactly ONE variable"
for name, cfg in CONFIGS.items():
    (PROJECT_ROOT / "configs" / f"{name}.json").write_text(json.dumps(cfg, indent=2))

In [ ]:
# 4.4 Train / fine-tune each experiment (skipped if a checkpoint exists)
def train(exp_name: str, cfg: dict):
    ckpt = CKPT_DIR / exp_name
    if SKIP_TRAINING_IF_CKPT and ckpt.exists() and any(ckpt.iterdir()):
        print(f"[{exp_name}] checkpoint found, skipping training"); return ckpt
    set_seed(cfg["seed"])
    ids = train_subset(cfg["n_train_utts"])
    # TODO(M2/M3): build the dataset from ids, train for cfg['steps'], validate on SPLIT['val'], save best to ckpt
    raise NotImplementedError

# for name, cfg in CONFIGS.items(): train(name, cfg)

In [ ]:
# 4.5 Training curves (train + val loss per experiment) -> results/figures/loss_curves.png
# TODO(M3): read logs/<exp>/ and plot both experiments on the same axes

# 5. End-to-end inference: arbitrary text → .wav *(Rubric 1.2, 11 marks; owner: Member 2)*
**HD checklist:** ☐ one function / one CLI entry point ☐ any text works ☐ consistent output folder ☐ runs top-to-bottom

In [ ]:
_MODELS = {}

def load_model(exp: str):
    if exp not in _MODELS:
        # TODO(M2): load acoustic model + vocoder from CKPT_DIR / exp
        raise NotImplementedError
    return _MODELS[exp]

def synthesize(text: str, exp: str = "expB", out_path=None):
    # Returns (waveform float32, sr) and writes a 16-bit PCM wav if out_path is given
    model = load_model(exp)
    clean = normalise_text(text)
    wav = None  # TODO(M2): wav = model(clean)
    sr = AUDIO_CFG["sr"]
    if out_path is not None:
        out_path = Path(out_path); out_path.parent.mkdir(parents=True, exist_ok=True)
        sf.write(out_path, wav, sr, subtype="PCM_16")
    return wav, sr

# Demo: change the text and run
# wav, sr = synthesize("Text to speech on LJSpeech, built by our group.", exp="expB", out_path=WAV_OUT_DIR / "demo.wav")
# display(Audio(wav, rate=sr))

In [ ]:
# 5.1 Qualitative prompts (>=10, different sentences; includes hard cases for §7)
PROMPTS = [
    "The quick brown fox jumps over the lazy dog.",
    "<everyday sentence>",
    "<question?>",
    "<long sentence with commas, clauses, and a final exclamation!>",
    "<numbers: On 2 November 2026 at 11:59 pm, 3.5 percent ...>",
    "<abbreviations: Dr. Smith met Mr. Jones at St. Peter's.>",
    "<rare names / place names>",
    "<out-of-domain technical words>",
    "<very short: Yes.>",
    "<sentence from the test split (to compare with ground truth)>",
]
assert len(PROMPTS) >= 10

def generate_examples(exp):
    paths = []
    for i, t in enumerate(PROMPTS, 1):
        p = WAV_OUT_DIR / f"{exp}_{i:02d}.wav"
        synthesize(t, exp=exp, out_path=p); paths.append(p)
    return paths

# for exp in CONFIGS: generate_examples(exp)
_ = (WAV_OUT_DIR / "prompts.txt").write_text("\n".join(f"{i:02d}\t{t}" for i, t in enumerate(PROMPTS, 1)) + "\n")

# 6. Objective evaluation *(Rubric 1.4, 6 marks; report 2.4; owner: Member 3)*
**HD checklist:** ☐ metric per experiment ☐ warm-up + repeated runs ☐ mean ± std ☐ device stated ☐ interpreted, not just reported

| Metric | What it measures | Direction |
|---|---|---|
| **RTF** = synth time / audio duration | speed (deployability) | lower = better, <1 = real-time |
| **Latency** (s / sentence) | user-perceived wait | lower |
| **WER / CER** via Whisper ASR | intelligibility | lower |
| **MCD** (dB) vs ground truth (test split) | spectral distance | lower |
| **UTMOS** (optional) | predicted naturalness MOS | higher |

In [ ]:
# 6.1 Speed: RTF and latency with warm-up and repeated runs
def measure_speed(exp, texts, n_warmup=2, n_runs=5):
    rows = []
    for _ in range(n_warmup):
        synthesize(texts[0], exp=exp)
    for t in texts:
        times, dur = [], None
        for _ in range(n_runs):
            t0 = time.perf_counter()
            wav, sr = synthesize(t, exp=exp)
            times.append(time.perf_counter() - t0)
            dur = len(wav) / sr
        rows.append(dict(exp=exp, text=t, latency_s=np.mean(times), latency_std=np.std(times),
                         audio_s=dur, rtf=np.mean(times) / dur))
    return pd.DataFrame(rows)

DEVICE = "<cpu / mps / cuda:0>"   # TODO: state the device used for timing
# speed = pd.concat([measure_speed(e, PROMPTS) for e in CONFIGS])

In [ ]:
# 6.2 Intelligibility: WER/CER of Whisper transcriptions of synthesised TEST sentences
# Whisper: Radford et al. (2022) https://github.com/openai/whisper ; jiwer for WER https://github.com/jitsi/jiwer
# TODO(M3)

In [ ]:
# 6.3 Quality: MCD vs ground truth on N test utterances (with DTW alignment), optional UTMOS
# MCD: Kubichek (1993). Implementation reference: TODO(M3) add the link to the library/code you adapt
# TODO(M3)

In [ ]:
# 6.4 Results table -> results/metrics.csv (this exact table goes in the report)
# summary = speed.groupby("exp").agg(rtf_mean=("rtf","mean"), rtf_std=("rtf","std"), latency_mean=("latency_s","mean"))
# summary = summary.join(wer_df).join(mcd_df)
# summary["device"] = DEVICE; summary.to_csv(RESULTS_DIR / "metrics.csv"); summary

In [ ]:
# 6.5 Trade-off plot: speed vs intelligibility (one point per experiment) -> results/figures/tradeoff.png
# TODO(M3)

# 7. Qualitative evidence & error analysis *(report 2.4 / 2.5)*
## 7.1 Listening panel: expA vs expB, same prompt

In [ ]:
# for i, t in enumerate(PROMPTS, 1):
#     print(f"{i:02d}. {t}")
#     for e in CONFIGS:
#         print("  ", e); display(Audio(str(WAV_OUT_DIR / f"{e}_{i:02d}.wav")))

In [ ]:
# 7.2 Spectrograms: ground truth vs expA vs expB for 2-3 test utterances -> results/figures/spec_compare_*.png
# TODO(M3)

## 7.3–7.4 Failure modes (≥2, with wav references)
| Failure mode | Example wav | Likely cause | Proposed fix |
|---|---|---|---|
| `<word skipping / repetition>` | `expA_04.wav` | `<attention misalignment on long input>` | `<duration model / monotonic alignment>` |
| `<number/abbrev mispronounced>` | | `<front-end>` | `<better normaliser>` |
| `<flat prosody on questions>` | | | |

## 7.5 Blind A/B preference (beyond HD)
`<n listeners × 10 prompts, randomised order: expA preferred __%, expB __%, no preference __%>`

# 8. Recommendation, deployment & scaling *(report 2.5 + 2.1 feasibility)*
## 8.1 Recommended variant
`<expX>` because `<metric evidence>`.
## 8.2 Scaling estimates
- Training on all 11,790 train utterances ≈ `<time per epoch × epochs>` on `<device>`
- CPU deployment: RTF `<x>` → `<n>` concurrent real-time streams per core
- Model size: `<MB>` · memory at inference: `<MB>`

# 9. Attribution & AI use statement *(Rubric 1.4)*
| Resource | Used for | Link |
|---|---|---|
| LJSpeech 1.1 | dataset | https://keithito.com/LJ-Speech-Dataset/ |
| `<toolkit>` | `<...>` | `<...>` |
| `<pretrained checkpoint>` | `<...>` | `<...>` |

**AI use statement:** `<tools, what for, example prompt; what we verified ourselves>`

# Final verification
☐ Restart & Run All passes · ☐ `wav_examples/` has ≥10 files per experiment · ☐ `results/metrics.csv` exists · ☐ no absolute paths · ☐ `requirements.txt` up to date